# 04 Hugging Face 기반 AI 에이전트 사례

## 04-2 멀티 태스크 기반 AI 에이전트 사례

### 04-2-4 실습: 멀티 태스크 AI 에이전트 실행

> 실습 목적 “여러 모델을 동시에 쓰는 것”이 아니라 “모델 호출 순서가 에이전트 구조임을 체감  

#### (1) 실습 1. 감성 분류 → 요약 순차 실행  

##### 1) Hugging Face transformers 라이브러리 설치

In [1]:
# -q 옵션: 설치 로그를 간단히 표시
!pip install transformers -q


##### 2) Hugging Face 토큰 발급 및 Colab 보안 저장소에 추가

i.  **Hugging Face 웹사이트에서 토큰 발급**: Hugging Face (huggingface.co) 에 로그인하여 `Settings` -> `Access Tokens` 페이지에서 새 토큰을 생성합니다. (권한은 `read` 이상으로 설정)  

ii.  **Colab 보안 저장소에 추가**: Colab 환경에서 왼쪽 패널의 '🔑' 아이콘(비밀번호 모양)을 클릭하여 `Secret` 탭을 엽니다. `New secret` 버튼을 클릭하여 `Name`에 `HF_TOKEN`을 입력하고, `Value`에 Hugging Face에서 발급받은 토큰 값을 붙여넣습니다. 그리고 'Notebook access'를 켜주세요.

##### 3) 코드에서 Hugging Face 토큰 사용하기

이제 Colab 보안 저장소에 저장된 `HF_TOKEN`을 코드에서 불러와 Hugging Face에 로그인할 수 있습니다. 아래 코드를 실행해 주세요.

In [3]:
from huggingface_hub import login
from google.colab import userdata

# Colab 보안 저장소에서 HF_TOKEN 불러오기
hf_token = userdata.get('HF_TOKEN')


# Hugging Face 로그인
login(token=hf_token)

print("Hugging Face에 성공적으로 로그인했습니다!")

Hugging Face에 성공적으로 로그인했습니다!


##### 4) 멀티 태스크 조합

1. 원문 대상 감성분류  
2. 원문 대상 Q&A
3. 원문 대상 요약
4. 요약문 대상 감성분류  
5. 요약문 대상 Q&A  



In [101]:
from transformers import pipeline

# 1. 감성 분류 모델 정의
classifier = pipeline(
    task="text-classification",
    model="nlptown/bert-base-multilingual-uncased-sentiment"
)

# 2. Q&A 모델 정의
# 한국어도 어느 정도 되는 다국어 모델 예시: deepset/xlm-roberta-base-squad2
# (영어 전용이면: deepset/roberta-base-squad2 등)
qa = pipeline(
    task="question-answering",
    model="deepset/xlm-roberta-base-squad2",
    tokenizer="deepset/xlm-roberta-base-squad2",
)

# 3. 요약 모델 정의
summarizer = pipeline(
    task="summarization",
    model="gogamza/kobart-base-v2",
    tokenizer="gogamza/kobart-base-v2"
)

review = """
제품의 품질은 우수하며, 기대했던 기준을 충족합니다.
사용된 소재와 전반적인 마감은 신뢰할 수 있고 잘 설계된 느낌을 줍니다.
그러나 배송 과정은 약속되었던 것보다 훨씬 느렸습니다.
택배는 예상된 날짜보다 며칠 늦게 도착했습니다.
배송 지연에 대해 사전 안내나 명확한 설명이 전혀 없었습니다.
이로 인해 기다리는 동안 불편함과 불확실함을 겪어야 했습니다.
또한 고객 지원에 문의했을 때 도움이 되지 않았습니다.
응답은 느렸고, 유용한 정보를 제공하지 못했습니다.
명확한 소통이 부족했던 점은 실망스러웠습니다.
배송 속도와 고객 지원을 개선한다면 전반적인 이용 경험이 크게 향상될 것입니다.
"""

config.json:   0%|          | 0.00/953 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/669M [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/39.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

Device set to use cpu
Device set to use cpu
You passed `num_labels=3` which is incompatible to the `id2label` map of length `2`.
You passed `num_labels=3` which is incompatible to the `id2label` map of length `2`.
You passed `num_labels=3` which is incompatible to the `id2label` map of length `2`.
You passed `num_labels=3` which is incompatible to the `id2label` map of length `2`.
Device set to use cpu


###### 1. 원문 대상 감성분류

In [102]:
# 원문에 대한 감성 분류
sentiment = classifier(review)
print("\n\n--- 원문 대상 감성분석 결과 ---\n")
print("Sentiment:", sentiment)



--- 원문 대상 감성분석 결과 ---

Sentiment: [{'label': '2 stars', 'score': 0.5934934616088867}]


###### 2. 원문 대상 Q&A

In [103]:
 # 지문(context)과 질문(question)

 question = "제품 리뷰의 주요 불만요소는 무엇인가?"

 # 실행
 result = qa(question=question, context=review)

 # 결과 출력
 print("Answer :", result["answer"])
 print("Score  :", round(float(result["score"]), 4))
 print("Span   :", (result["start"], result["end"]))

Answer : 
배송 지연에
Score  : 0.0256
Span   : (129, 136)


###### 3. 원문 대상 요약

In [104]:
import re
from transformers import pipeline

def preprocess_text(text):
    # Only remove numbers and multiple spaces from the input review
    text = re.sub(r'\d+', '', text)        # Remove numbers
    text = re.sub(r'\s+', ' ', text)       # Replace multiple spaces with a single space
    text = text.strip()
    return text

def postprocess_summary(summary_text):
    # 1. Ensure a space after punctuation for better sentence splitting and readability
    summary_text = re.sub(r'([.?!])(?=\S)', r'\1 ', summary_text) # Use positive lookahead to avoid double spaces

    # 2. Remove consecutive identical words (e.g., "없었 없었습니다")
    summary_text = re.sub(r'(\S+)\s+\1', r'\1', summary_text)

    # 3. Find the first full sentence (ending with a period, question mark, or exclamation mark)
    sentences = re.split(r'([.?!])\s*', summary_text)
    cleaned_sentences = [s.strip() for s in sentences if s.strip()]

    if cleaned_sentences:
        first_sentence_base = cleaned_sentences[0]

        # Remove leading particles (like '은/는/이/가') from the base sentence part
        first_sentence_cleaned_particles = re.sub(r'^[은는이가을를]​*\s*', '', first_sentence_base)

        # Apply specific heuristic for the "missing subject" case '신뢰하며'
        final_first_sentence = first_sentence_cleaned_particles
        if final_first_sentence.startswith('신뢰하며') and not final_first_sentence.startswith('제품은'):
            final_first_sentence = '제품은 ' + final_first_sentence

        # Reconstruct the first sentence, including its punctuation if any
        if len(cleaned_sentences) > 1 and cleaned_sentences[1] in ['.', '?', '!']:
            final_first_sentence += cleaned_sentences[1]

        # Concatenate the remaining sentences, ensuring spaces between them
        remaining_summary_parts = [s for i, s in enumerate(cleaned_sentences) if i >= 2 and s not in ['.', '?', '!']]
        remaining_summary = ' '.join(remaining_summary_parts)

        # Return the processed first sentence plus the remaining summary
        if remaining_summary:
            return final_first_sentence + ' ' + remaining_summary
        else:
            return final_first_sentence
    return summary_text


# 원본 리뷰 텍스트 전처리
preprocessed_review = preprocess_text(review)
print("\n--- 전처리된 원문 ---\n\n")
print(preprocessed_review)

# 전처리된 텍스트로 요약 실행
summary = summarizer(
    preprocessed_review,
    max_new_tokens=120,  # 적절한 최대 토큰 수 설정 (증가)
    min_length=10,       # 최소 길이 설정
    do_sample=False,
    num_beams=8,         # 더 나은 품질을 위한 빔 탐색 사용 (증가)
    early_stopping=True, # `eos_token_id`가 생성되면 빔 탐색 중지
    no_repeat_ngram_size=5 # 5-단어 시퀀스 반복 방지
)

# 요약 결과 후처리
final_summary = postprocess_summary(summary[0]["summary_text"])

# 결과 출력
print("\n\n--- 텍스트 요약 결과 ---\n")
print(final_summary)



--- 전처리된 원문 ---


제품의 품질은 우수하며, 기대했던 기준을 충족합니다. 사용된 소재와 전반적인 마감은 신뢰할 수 있고 잘 설계된 느낌을 줍니다. 그러나 배송 과정은 약속되었던 것보다 훨씬 느렸습니다. 택배는 예상된 날짜보다 며칠 늦게 도착했습니다. 배송 지연에 대해 사전 안내나 명확한 설명이 전혀 없었습니다. 이로 인해 기다리는 동안 불편함과 불확실함을 겪어야 했습니다. 또한 고객 지원에 문의했을 때 도움이 되지 않았습니다. 응답은 느렸고, 유용한 정보를 제공하지 못했습니다. 명확한 소통이 부족했던 점은 실망스러웠습니다. 배송 속도와 고객 지원을 개선한다면 전반적인 이용 경험이 크게 향상될 것입니다.


--- 텍스트 요약 결과 ---

제품은 신뢰하며, 기대했던 기준을 충족합니다. 사용된 소재와 전반적인 마감은 신뢰할 수 있고 잘 설계된 느낌을 줍니다 그러나 배송 과정은 약속되었던 것보다 훨씬 느렸습니다 택배는 예상된 날짜보다 며칠 늦게 도착했습니다 배송 지연에 대해 사전 안내나 명확한 설명이 전혀 없었습니다 이로 인해 기다리는 동안 불편함과 불확실함을 겪어야 했습니다 또한 고객 지원에 문의했을 때 도움이 되지 않았습니다 응답은 느렸고, 유용한 정보를 제공 제공이 못했습니다 명확한 소통이 부족했던 점은 실망스러웠습니다 배송 속도와 고객 지원을 개선하며, 기대했던 정보를 충족하며, 기대


###### 4. 요약문 대상 감성분석

In [105]:
# 요약에 대한 감성분류
# sentiment = classifier(preprocessed_review)
sentiment = classifier(final_summary)
print("\n\n--- 요약 대상 감성분석 결과 ---\n")
print("Sentiment:", sentiment)



--- 요약 대상 감성분석 결과 ---

Sentiment: [{'label': '2 stars', 'score': 0.49756503105163574}]


###### 5. 요약문 대상 Q&A

In [106]:
# 지문(context)과 질문(question)

question = "제품 리뷰의 주요 불만요소는 무엇인가?"

# 실행
result = qa(question=question, context=final_summary)

# 결과 출력
print("Answer :", result["answer"])
print("Score  :", round(float(result["score"]), 4))
print("Span   :", (result["start"], result["end"]))

Answer :  불확실함을
Score  : 0.0472
Span   : (174, 180)


- 관찰 포인트  
    - 두 모델의 출력 역할이 명확히 분리되는가?  
    - 요약 후 감성분류 결과가 달라질까?  

#### (2) 실습 2. 요약 → 질의응답 흐름 설계 (개념 실습)  

-  요약 결과를 QA 모델의 context로 사용한다고 가정  
- “이 구조가 왜 필요한가?” 토의  